In [3]:
import os
import re
import json
import glob
import hashlib
import asyncio
from pathlib import Path
from typing import TypedDict, List, Dict, Any, Optional, Iterable
from datetime import datetime
from collections import defaultdict

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=False)

from langgraph.graph import StateGraph, END
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings

# Optional (Jupyter)
try:
    from IPython.display import HTML, display
except Exception:
    def display(*_a, **_k): pass
    class HTML:  # type: ignore
        def __init__(self, *_a, **_k): pass

# MCP client
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport

# Pretty table
import pandas as pd
from tabulate import tabulate

# =========================
# === CONFIG (ENV) ========
# =========================
# ---- RAG ----
JSON_INPUT = os.getenv("EXPORT_JSONL_OUTPUT", "./datahub_metadata/*.jsonl").strip()
CHROMA_DIR = os.getenv("CHROMA_DIR", "./chroma_db")
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "classicmodels")
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text")
OLLAMA_BASE_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
CHAT_MODEL = os.getenv("CHAT_MODEL", "qwen2.5:3b")
TOP_K = int(os.getenv("TOP_K", "5"))

RETRIEVAL_PROBE = (
    " measure measures metric metrics sum total amount price revenue sales gross net "
    "quantity qty count average avg mean distinct group by bucket date time timestamp "
    "created updated order_date month week quarter year category region customer status id"
)

# ---- MCP ----
MCP_BASE    = os.getenv("MCP_BASE", "http://mcp_server:8000/mcp")
MCP_TIMEOUT = int(os.getenv("MCP_CLIENT_TIMEOUT", "60"))

# ---- Export ----
EXPORT_DIR = os.getenv("EXPORT_DIR", "exports")
JUPYTER_BASE_URL = os.getenv("JUPYTER_BASE_URL", "").rstrip("/")

# ---- Debug flag (0=off, 1=on) ----
SQL_DEBUG = int(os.getenv("SQL_DEBUG", "1"))
def _dbg_print_two(label: str, sql: str):
    if SQL_DEBUG:
        print(f"\n{label}")
        print(sql)

# =========================
# === RAG (STATE) =========
# =========================
class RAGState(TypedDict):
    jsonl_paths: List[str]
    question: str
    added: int
    updated: int
    skipped: int
    context: str
    answer: str
    draft_sql: str
    final_sql: str

def resolve_jsonl_paths(spec: str) -> List[str]:
    paths: List[str] = []
    if not spec:
        return paths
    p = Path(spec)
    if any(ch in spec for ch in "*?[]"):
        paths.extend(glob.glob(spec))
    elif p.is_dir():
        paths.extend(str(x) for x in p.glob("*.jsonl"))
    elif p.is_file():
        paths.append(str(p))
    else:
        paths.extend(glob.glob(spec))
    out, seen = [], set()
    for x in paths:
        xp = str(Path(x))
        if xp.lower().endswith(".jsonl") and xp not in seen:
            seen.add(xp)
            out.append(xp)
    return out

def build_vector_store() -> Chroma:
    embeddings = OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_BASE_URL)
    return Chroma(
        persist_directory=CHROMA_DIR,
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
    )

def read_jsonl(path: str) -> Iterable[tuple[str, Optional[Dict[str, Any]]]]:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            try:
                obj = json.loads(s)
            except Exception:
                obj = None
            yield s, obj

def _safe_iso_to_dt(s: Optional[str]) -> Optional[datetime]:
    if not s:
        return None
    try:
        return datetime.fromisoformat(s)
    except Exception:
        return None

def _stable_id(meta: Dict[str, Any], source_name: str) -> str:
    key = f"{meta.get('database','')}|{meta.get('table','')}|{meta.get('column','')}|{source_name}"
    return hashlib.sha1(key.encode("utf-8")).hexdigest()

def show_graph_mermaid_only(app) -> None:
    try:
        G = app.get_graph()
        mermaid_src = G.draw_mermaid()
    except Exception:
        return
    html = f"""
<div class="mermaid">
{mermaid_src}
</div>
<script>
(function () {{
  function renderMermaid() {{
    if (window.mermaid) {{
      mermaid.initialize({{ startOnLoad: true, securityLevel: 'loose' }});
      mermaid.init();
    }}
  }}
  if (!window.mermaid) {{
    var s = document.createElement('script');
    s.src = 'https://cdn.jsdelivr.net/npm/mermaid/dist/mermaid.min.js';
    s.onload = renderMermaid;
    document.body.appendChild(s);
  }} else {{
    renderMermaid();
  }}
}})();
</script>
"""
    display(HTML(html))

# =========================
# === SQL-AWARE CARDS =====
# =========================
def _column_card(meta: Dict[str, Any]) -> str:
    db = meta.get("database", "") or meta.get("db", "")
    sch = meta.get("schema", "") or meta.get("schemaName", "")
    tbl = meta.get("table", "") or meta.get("tableName", "")
    col = meta.get("column", "") or meta.get("columnName", "")
    dtype = meta.get("dtype") or meta.get("dataType") or meta.get("type", "")
    desc = meta.get("description") or meta.get("brief", "") or ""
    aliases = meta.get("aliases") or meta.get("synonyms") or meta.get("aka") or []
    if isinstance(aliases, str):
        aliases = [aliases]
    sample_vals = meta.get("sample_values") or meta.get("samples") or meta.get("examples") or []
    if isinstance(sample_vals, dict):
        sample_vals = list(sample_vals.values())
    sample_txt = ", ".join(map(lambda x: str(x)[:40], sample_vals[:5]))

    fqn = ".".join([p for p in [db, sch, tbl, col] if p])
    return (
        f"TABLE: {tbl}\n"
        f"COLUMN: {col}\n"
        f"TYPE: {dtype}\n"
        f"ALIASES: {', '.join(map(str, aliases))}\n"
        f"DESC: {desc}\n"
        f"SAMPLE: {sample_txt}\n"
        f"FQN: {fqn}"
    ).strip()

# =========================
# === PIPELINE NODES ======
# =========================
def ingest_node(state: RAGState) -> RAGState:
    paths = state["jsonl_paths"]
    vs = build_vector_store()

    total_in_collection = vs._collection.count()
    first_run = (total_in_collection == 0)
    if first_run:
        print("First run detected (empty collection) - embedding all documents.")

    added = updated = skipped = 0

    for path in paths:
        p = Path(path)
        if not p.exists():
            print(f"JSONL not found: {path}")
            continue

        docs: List[Document] = []
        ids: List[str] = []
        metas: List[Dict[str, Any]] = []

        for raw, obj in read_jsonl(path):
            meta: Dict[str, Any] = obj if isinstance(obj, dict) else {"raw": raw}
            meta["_source"] = p.name
            metas.append(meta)
            ids.append(_stable_id(meta, p.name))
            # <<< use SQL-aware column cards >>>
            docs.append(Document(page_content=_column_card(meta), metadata=meta))

        if not docs:
            continue

        if first_run:
            vs.add_documents(docs, ids=ids)
            added += len(docs)
            print(f"Embedded {len(docs)} docs from {path}")
            continue

        existing_meta_by_id: Dict[str, Dict[str, Any]] = {}
        BATCH = 1000
        for i in range(0, len(ids), BATCH):
            chunk_ids = ids[i:i + BATCH]
            got = vs._collection.get(ids=chunk_ids, include=["metadatas"])
            got_ids = got.get("ids") or []
            got_metas = got.get("metadatas") or []
            for doc_id, md in zip(got_ids, got_metas):
                existing_meta_by_id[doc_id] = md or {}

        to_upsert_docs: List[Document] = []
        to_upsert_ids: List[str] = []

        for doc, doc_id, meta in zip(docs, ids, metas):
            incoming_li = meta.get("last_ingested")
            existing = existing_meta_by_id.get(doc_id)

            if not existing:
                to_upsert_docs.append(doc)
                to_upsert_ids.append(doc_id)
                continue

            stored_li = existing.get("last_ingested")
            should_update = (incoming_li != stored_li)

            in_dt = _safe_iso_to_dt(incoming_li)
            st_dt = _safe_iso_to_dt(stored_li)
            if in_dt and st_dt:
                should_update = in_dt > st_dt

            if should_update:
                to_upsert_docs.append(doc)
                to_upsert_ids.append(doc_id)
            else:
                skipped += 1

        if to_upsert_docs:
            vs.delete(ids=to_upsert_ids)
            vs.add_documents(to_upsert_docs, ids=to_upsert_ids)
            updated += len(to_upsert_docs)
            print(f"Upserted {len(to_upsert_docs)} changed docs from {path}")

    try:
        vs.persist()
    except Exception:
        pass

    state["added"] = added
    state["updated"] = updated
    state["skipped"] = skipped
    return state

def retrieve_node(state: RAGState) -> RAGState:
    vs = build_vector_store()
    retriever = vs.as_retriever(search_kwargs={"k": max(3, TOP_K)})
    q = (state["question"] or "").strip()
    hybrid_query = (q + " " + RETRIEVAL_PROBE).strip()
    docs = retriever.invoke(hybrid_query)

    # Build per-table whitelist + alias map from top docs
    allowed = defaultdict(list)      # table -> [cols]
    alias_map = defaultdict(dict)    # table -> alias(lower) -> canonical
    for d in (docs or []):
        md = d.metadata or {}
        t = (md.get("table") or md.get("tableName") or "").strip()
        c = (md.get("column") or md.get("columnName") or "").strip()
        if t and c:
            allowed[t].append(c)
            aliases = md.get("aliases") or md.get("synonyms") or []
            if isinstance(aliases, str):
                aliases = [aliases]
            for al in aliases:
                if al:
                    alias_map[t][str(al).lower()] = c

    # Heuristic: narrow likely anchor tables by question tokens (keeps if matched)
    q_l = q.lower()
    likely_tables = set()
    if any(w in q_l for w in ["customer", "customers", "customer name", "customer country"]):
        likely_tables.add("customers")
    if any(w in q_l for w in ["order", "orders", "order date", "order status"]):
        likely_tables.add("orders")
    if any(w in q_l for w in ["product", "products", "productline", "product line"]):
        likely_tables.add("products")
    if likely_tables:
        allowed = {t: cols for t, cols in allowed.items() if t.lower() in likely_tables}
    else:
        allowed = dict(allowed)

    allowed_lines = []
    for t, cols in allowed.items():
        uniq = sorted({*cols})
        allowed_lines.append(f"- {t}: {', '.join(uniq)}")
    allowed_block = "### ALLOWED IDENTIFIERS\n" + ("\n".join(allowed_lines) if allowed_lines else "- (none)")

    cards_text = "\n".join(d.page_content for d in (docs or []))
    context = (allowed_block + "\n\n" + cards_text).strip()
    context += f"\n\n### _INTERNAL_HINT\n{json.dumps({'alias_map': alias_map}, ensure_ascii=False)}"

    state["context"] = context
    return state

# =========================
# === Sanitizer & SQL PP ==
# =========================
_AGG_WORDS = re.compile(r"\b(sum|avg|count|min|max)\s*\(", re.I)
JOIN_KW = r"(?:inner\s+join|left\s+join|right\s+join|full\s+join|join)"

def _normalize_cols(s: str) -> List[str]:
    parts = [p.strip() for p in s.split(",")]
    out = []
    for p in parts:
        m = re.split(r"\s+AS\s+", p, flags=re.I)
        out.append(m[0].strip())
    return out

def _has_join(sql: str) -> bool:
    return re.search(rf"\b{JOIN_KW}\b", sql, flags=re.I) is not None

def _prefer_distinct_over_group_by(sql: str) -> str:
    m = re.search(r"^\s*select\s+(.*?)\s+from\s+(.*?)(\s+group\s+by\s+(.*?))(?:\s+order\s+by\s+.*)?;", sql, flags=re.I | re.S)
    if not m:
        return sql
    select_cols = _normalize_cols(m.group(1))
    group_by_cols = _normalize_cols(m.group(4))
    if _AGG_WORDS.search(sql):
        return sql
    if select_cols and group_by_cols and set(map(str.lower, select_cols)) == set(map(str.lower, group_by_cols)):
        sql2 = re.sub(r"^\s*select\s+", "SELECT DISTINCT ", sql, flags=re.I)
        sql2 = re.sub(r"\s+group\s+by\s+.*?(?=(\s+order\s+by\s+.*)?;)", "", sql2, flags=re.I | re.S)
        return sql2
    return sql

def _split_qualified(name: str) -> List[str]:
    return [p.strip('`" ') for p in name.split(".") if p.strip("`\" ")]

def _short_table(name_or_fqn: str) -> str:
    parts = _split_qualified(name_or_fqn)
    return parts[-1] if parts else name_or_fqn.strip('`"')

# ===== Alias handling (for sanitizer) =====
def _parse_from_table_alias(sql: str) -> Optional[tuple[str, Optional[str]]]:
    m = re.search(
        r"\bfrom\s+([`\"\w\.]+)(?:\s+(?:AS\s+)?([`\"\w]+))?",
        sql, flags=re.I
    )
    if not m:
        return None
    table_fqn = m.group(1)
    alias = m.group(2)
    return table_fqn, alias

def _build_alias_table_map(sql: str) -> Dict[str, str]:
    amap: Dict[str, str] = {}
    def _add(table_fqn: str, alias: Optional[str]):
        tshort = _short_table(table_fqn)
        if alias:
            amap[alias.strip('`"')] = tshort
        amap[tshort] = tshort
    fm = _parse_from_table_alias(sql)
    if fm: _add(fm[0], fm[1])
    for jm in re.finditer(
        rf"\b{JOIN_KW}\s+([`\"\w\.]+)(?:\s+(?:AS\s+)?([`\"\w]+))?",
        sql, flags=re.I
    ):
        _add(jm.group(1), jm.group(2))
    return amap

def _should_use_backtick(identifier: str) -> bool:
    if not identifier: return False
    clean = identifier.strip('`"')
    if not clean: return False
    first_char = clean[0]
    if first_char.isupper() or first_char.isdigit():
        return True
    if not re.match(r'^[a-z_][a-z0-9_]*$', clean, re.I):
        return True
    return False

def _smart_backtick(identifier: str) -> str:
    clean = identifier.strip('`"')
    return f"`{clean}`" if _should_use_backtick(clean) else clean

def _fix_bad_backticks(sql: str) -> str:
    def repl_fqn(m: re.Match) -> str:
        inside = m.group(1)
        if "." not in inside:
            return _smart_backtick(inside)
        parts = _split_qualified(inside)
        if len(parts) == 0:
            return inside
        return ".".join(_smart_backtick(p) for p in parts)
    sql = re.sub(r"`([^`]+?\.[^`]+?)`", repl_fqn, sql)
    sql = re.sub(r"(\b[A-Za-z_][\w]*)\.`([^`]+?)`",
                 lambda m: f"{_smart_backtick(m.group(1))}."
                           + ".".join(_smart_backtick(p) for p in _split_qualified(m.group(2))), sql)
    sql = re.sub(r"`([^`]+?)`\.(\b[A-Za-z_][\w]*)",
                 lambda m: ".".join(_smart_backtick(p) for p in _split_qualified(m.group(1)))
                           + f".{_smart_backtick(m.group(2))}", sql)
    sql = re.sub(r'`([^`]+?)`(?!\s*\.)', lambda m: _smart_backtick(m.group(1)), sql, flags=re.I)
    return sql

def _select_qualifiers(sql: str) -> set:
    m = re.search(r"^\s*select\s+(.*?)\s+from\s+", sql, flags=re.I | re.S)
    if not m:
        return set()
    select = m.group(1)
    quals = set()
    for q in re.findall(r"\b([A-Za-z_][A-Za-z0-9_]*)\s*\.", select):
        quals.add(q.strip('`"'))
    for sch, tbl, col in re.findall(r"`([^`]+)`\s*\.\s*`([^`]+)`\s*\.\s*`([^`]+)`", select):
        quals.add(tbl)
    for tbl, col in re.findall(r"`([^`]+)`\s*\.\s*`([^`]+)`", select):
        quals.add(tbl)
    return quals

def _has_filters(sql: str) -> bool:
    return bool(re.search(r"\b(where|group\s+by|having)\b", sql, flags=re.I))

def _strip_all_joins_unconditionally(sql: str) -> str:
    return re.sub(
        rf"\s+{JOIN_KW}\s+[`\"\w\.]+(?:\s+[`\"\w]+)?\s+on\s+[^;]+?(?=\b({JOIN_KW}|where|group|order|limit|having|;)\b)",
        "",
        sql,
        flags=re.I | re.S
    )

def _find_anchor_qualifier(sql: str) -> Optional[str]:
    fm = _parse_from_table_alias(sql)
    if not fm:
        return None
    table_fqn, alias = fm
    return (alias or _short_table(table_fqn)).strip('`"')

def _remove_redundant_joins(sql: str) -> str:
    pattern = re.compile(
        rf"\b({JOIN_KW})\s+([`\"\w\.]+)(?:\s+([`\"\w]+))?\s+on\s+([^;]+?)(?=\b({JOIN_KW}|where|group|order|limit|;)\b)",
        re.I | re.S
    )
    joins = list(pattern.finditer(sql))
    if not joins:
        return sql

    def alias_used_outside_on(qualifier: str, sql_text: str, on_span: tuple[int,int], table_name_short: str) -> bool:
        before = sql_text[:on_span[0]]
        after  = sql_text[on_span[1]:]
        pat1 = re.compile(rf"\b{re.escape(qualifier)}\s*\.", re.I)
        pat2 = re.compile(rf"\b{re.escape(table_name_short)}\s*\.", re.I)
        return bool(pat1.search(before) or pat1.search(after) or pat2.search(before) or pat2.search(after))

    new_sql = sql
    offset = 0
    for j in joins:
        table_fqn = j.group(2).strip('`"')
        alias = j.group(3).strip('`"') if j.group(3) else None
        qualifier = alias or _short_table(table_fqn)
        on_start = j.start(4) + offset
        on_end   = j.end(4) + offset
        block_span = (j.start() + offset, j.end() + offset)
        used = alias_used_outside_on(qualifier, new_sql, (on_start, on_end), _short_table(table_fqn))
        if not used:
            new_sql = new_sql[:block_span[0]] + new_sql[block_span[1]:]
            offset -= (block_span[1] - block_span[0])
    return new_sql

def _strip_on_clauses(sql: str) -> str:
    return re.sub(rf"\bon\s+([^;]+?)(?=\b({JOIN_KW}|where|group|order|limit|;)\b)", " ON ", sql, flags=re.I | re.S)

def _qualifiers_used_outside_on(sql: str) -> set:
    stripped = _strip_on_clauses(sql)
    quals = set(m.group(1).strip('`"') for m in re.finditer(r"\b([`\"\w]+)\s*\.", stripped))
    return set(q.strip('`"') for q in quals if q)

def _pushdown_filters_through_equi_join(sql: str) -> str:
    anchor = _find_anchor_qualifier(sql)
    if not anchor:
        return sql

    pattern = re.compile(
        rf"\b({JOIN_KW})\s+([`\"\w\.]+)(?:\s+([`\"\w]+))?\s+on\s+([^;]+?)(?=\b({JOIN_KW}|where|group|order|limit|;)\b)",
        re.I | re.S
    )
    alias_map: Dict[str, str] = {}
    joins = list(pattern.finditer(sql))
    for j in joins:
        table_fqn = j.group(2).strip('`"')
        alias = j.group(3).strip('`"') if j.group(3) else None
        qualifier = alias or _short_table(table_fqn)
        on_text = j.group(4)

        for m in re.finditer(r"\b([`\"\w]+)\.([`\"\w]+)\s*=\s*([`\"\w]+)\.([`\"\w]+)\b", on_text):
            a1, c1, a2, c2 = (m.group(1).strip('`"'), m.group(2).strip('`"'),
                               m.group(3).strip('`"'), m.group(4).strip('`"'))
            if a1 == anchor and a2 in {qualifier, _short_table(table_fqn)}:
                alias_map[f"{qualifier}.{c2}"] = f"{anchor}.{c1}"
            elif a2 == anchor and a1 in {qualifier, _short_table(table_fqn)}:
                alias_map[f"{qualifier}.{c1}"] = f"{anchor}.{c2}"

    if not alias_map:
        return sql

    new_sql = sql
    for j in joins:
        on_span = (j.start(4), j.end(4))
        before = new_sql[:on_span[0]]
        onpart = new_sql[on_span[0]:on_span[1]]
        after  = new_sql[on_span[1]:]
        for alias_col, anchor_col in alias_map.items():
            qname, col = alias_col.split(".")
            alias_regex = re.compile(rf"\b{re.escape(qname)}\s*\.\s*{re.escape(col)}\b", re.I)
            before = alias_regex.sub(anchor_col, before)
            after  = alias_regex.sub(anchor_col, after)
        new_sql = before + onpart + after

    new_sql = _remove_redundant_joins(new_sql)
    return new_sql

_IDLIKE = re.compile(r".*\b(id|.*_id|.*id$|.*number$|.*_number)\b", re.I)

def _select_has_anchor_id_like(sql: str, anchor: Optional[str]) -> bool:
    if not anchor:
        return False
    m = re.search(r"^\s*select\s+(.*?)\s+from\s+", sql, flags=re.I | re.S)
    if not m:
        return False
    cols = _normalize_cols(m.group(1))
    for col in cols:
        m2 = re.match(r"(?:(?:`?)([\w]+)(?:`?)\s*\.\s*(?:`?)([\w]+)(?:`?))", col)
        if not m2:
            continue
        qualifier, cname = m2.group(1), m2.group(2)
        if qualifier.strip('`"').lower() == anchor.lower() and _IDLIKE.match(cname):
            return True
    return False

def _drop_unnecessary_distinct(sql: str) -> str:
    if "distinct" not in sql.lower():
        return sql
    if _has_join(sql) or " group by " in sql.lower() or _AGG_WORDS.search(sql):
        return sql
    anchor = _find_anchor_qualifier(sql)
    if _select_has_anchor_id_like(sql, anchor):
        return re.sub(r"^\s*select\s+distinct\s+", "SELECT ", sql, flags=re.I)
    return sql

# ===== BETWEEN-aware tokenizer =====
def _tokenize_where_between_safe(where_body: str) -> List[str]:
    out: List[str] = []
    buf: List[str] = []
    i, n = 0, len(where_body)
    in_single = False
    paren = 0
    between_waits_for_and = False

    def is_word_char(ch: str) -> bool:
        return ch.isalnum() or ch == "_" or ch == "`"

    def at_word_boundary(start: int, word_len: int) -> bool:
        prev_ch = where_body[start - 1] if start > 0 else ""
        left_ok = not is_word_char(prev_ch)
        end = start + word_len
        next_ch = where_body[end] if end < n else ""
        right_ok = not is_word_char(next_ch)
        return left_ok and right_ok

    def flush():
        if buf:
            out.append("".join(buf))
            buf.clear()

    while i < n:
        ch = where_body[i]
        if ch == "'" and not in_single:
            in_single = True
            buf.append(ch); i += 1; continue
        elif ch == "'" and in_single:
            buf.append(ch); i += 1; in_single = False; continue

        if not in_single:
            if ch == "(":
                paren += 1
            elif ch == ")":
                paren = max(0, paren - 1)

            if paren == 0:
                if where_body[i:].lower().startswith("between") and at_word_boundary(i, 7):
                    between_waits_for_and = True
                    buf.append(where_body[i:i+7]); i += 7; continue
                if where_body[i:].lower().startswith("and") and at_word_boundary(i, 3):
                    if between_waits_for_and:
                        buf.append(where_body[i:i+3]); i += 3; between_waits_for_and = False; continue
                    else:
                        flush(); out.append("AND"); i += 3; continue
                if where_body[i:].lower().startswith("or") and at_word_boundary(i, 2):
                    flush(); out.append("OR"); i += 2; continue
        buf.append(ch); i += 1

    flush()
    out = [s.strip() for s in out if s is not None and s.strip() != ""]
    return out

def _cleanup_bool_expression(parts: List[str]) -> str:
    if not parts:
        return ""
    cleaned: List[str] = []
    idx = 0
    while idx < len(parts) and parts[idx].upper() in ("AND", "OR"):
        idx += 1
    last_was_conn = False
    while idx < len(parts):
        tok = parts[idx].strip()
        if tok.upper() in ("AND", "OR"):
            if last_was_conn or not cleaned:
                idx += 1; continue
            cleaned.append(tok.upper()); last_was_conn = True
        else:
            cleaned.append(tok); last_was_conn = False
        idx += 1
    while cleaned and cleaned[-1].upper() in ("AND", "OR"):
        cleaned.pop()
    out = []
    for t in cleaned:
        out.append(t.upper() if t.upper() in ("AND", "OR") else t)
    return " ".join(out).strip()

def _strip_where_non_anchor_refs(sql: str) -> str:
    anchor = _find_anchor_qualifier(sql)
    if not anchor:
        return sql
    m = re.search(r"\bwhere\b(.*?)(?=\border\b|\bgroup\b|\blimit\b|;|$)", sql, flags=re.I | re.S)
    if not m:
        return sql
    where_body = m.group(1)
    tokens = _tokenize_where_between_safe(where_body)

    kept: List[str] = []
    for tok in tokens:
        if tok.upper() in ("AND", "OR"):
            kept.append(tok.upper()); continue
        qualifiers = re.findall(r"\b([A-Za-z_][A-Za-z0-9_]*)\s*\.", tok)
        non_anchor_refs = [q for q in qualifiers if q.lower() != anchor.lower()]
        if non_anchor_refs:
            continue
        kept.append(tok)

    new_body = _cleanup_bool_expression(kept)
    start, end = m.span()
    if not new_body:
        return sql[:start] + sql[end:]
    tail = sql[end:]
    sep = "" if (not tail or tail[:1].isspace()) else " "
    return sql[:start] + " WHERE " + new_body + sep + tail

def _fix_missing_space_before_clauses(sql: str) -> str:
    return re.sub(
        r"([^\s])\b(ORDER\s+BY|GROUP\s+BY|HAVING|LIMIT|WHERE)\b",
        r"\1 \2",
        sql,
        flags=re.I
    )

def _strip_group_by_having_non_anchor(sql: str) -> str:
    anchor = _find_anchor_qualifier(sql)
    if not anchor:
        return sql
    m_gb = re.search(r"\bgroup\s+by\b(.*?)(?=\border\b|\blimit\b|\bhaving\b|;|$)", sql, flags=re.I | re.S)
    m_hv = re.search(r"\bhaving\b(.*?)(?=\border\b|\blimit\b|;|$)", sql, flags=re.I | re.S)

    def _clause_has_non_anchor_qual(body: str) -> bool:
        quals = set(q.strip('`"') for q in re.findall(r"\b([A-Za-z_][A-Za-z0-9_]*)\s*\.", body or ""))
        return any(q.lower() != (anchor or "").lower() for q in quals)

    if m_gb and _clause_has_non_anchor_qual(m_gb.group(1)):
        gb_start, gb_end = m_gb.span()
        if m_hv:
            hv_start, hv_end = m_hv.span()
            if gb_end <= hv_start:
                return sql[:gb_start] + sql[hv_end:]
        return sql[:gb_start] + sql[gb_end:]

    if m_hv and _clause_has_non_anchor_qual(m_hv.group(1)):
        hv_start, hv_end = m_hv.span()
        return sql[:hv_start] + sql[hv_end:]

    return sql

def _strip_lonely_having(sql: str) -> str:
    has_gb = bool(re.search(r"\bgroup\s+by\b", sql, flags=re.I))
    if has_gb: return sql
    return re.sub(r"\bhaving\b.*?(?=\border\b|\blimit\b|;|$)", "", sql, flags=re.I | re.S)

def _trim_dangling_closing_paren(sql: str) -> str:
    s = sql.rstrip()
    semi = s.endswith(";")
    if semi: s = s[:-1].rstrip()
    s = re.sub(r"\)+\s*$", "", s)
    return (s + ";") if semi else s

# ---------- LIMIT POLICY ----------
def _explicit_limit_from_question(question: str) -> Optional[int]:
    q = (question or "").lower()
    m = re.search(r"\blimit\s+(\d{1,4})\b", q)
    if m: return max(1, min(int(m.group(1)), 1_000_000))
    m = re.search(r"\btop\s+(\d{1,4})\b", q)
    if m: return max(1, min(int(m.group(1)), 1_000_000))
    m = re.search(r"\b(show|first|tampilkan|ambil)\s+(\d{1,4})\b", q)
    if m: return max(1, min(int(m.group(2)), 1_000_000))
    return None

def _maybe_add_limit(question: str, sql: str) -> str:
    if re.search(r"\blimit\s+\d+\b", sql, flags=re.I):
        return sql
    n = _explicit_limit_from_question(question)
    if n is None:
        return sql
    s = sql.rstrip().rstrip(";")
    return s + f" LIMIT {n};"

# === DISTINCT heuristics ===
def _wants_distinct_from_question(question: str) -> bool:
    q = (question or "").lower()
    return any(k in q for k in [
        "distinct", "unique", "no duplicate", "tanpa duplikat",
        "unik", "dedup", "de-dup", "de dup", "de dup", "hanya yang unik"
    ])

def _wants_all_rows_from_question(question: str) -> bool:
    q = (question or "").lower()
    return any(k in q for k in [
        "all rows", "include duplicates", "dengan duplikat",
        "raw rows", "semua baris", "tanpa dedup", "jangan distinct"
    ])

def _is_grouping_or_agg(sql: str) -> bool:
    s = sql.lower()
    return (" group by " in s) or (" having " in s) or bool(_AGG_WORDS.search(sql))

def _has_distinct(sql: str) -> bool:
    return bool(re.search(r"^\s*select\s+distinct\b", sql, flags=re.I))

def _add_distinct(sql: str) -> str:
    if _has_distinct(sql):
        return sql
    return re.sub(r"^\s*select\s+", "SELECT DISTINCT ", sql, flags=re.I)

def _remove_distinct(sql: str) -> str:
    return re.sub(r"^\s*select\s+distinct\s+", "SELECT ", sql, flags=re.I)

def _ensure_reasonable_distinct(sql: str, question: str) -> str:
    if not sql:
        return sql
    if _wants_all_rows_from_question(question):
        return _remove_distinct(sql)
    if _wants_distinct_from_question(question):
        return _add_distinct(sql)
    has_join = _has_join(sql)
    is_group_or_agg = _is_grouping_or_agg(sql)
    anchor = _find_anchor_qualifier(sql)
    if is_group_or_agg:
        return sql
    if not has_join:
        if _select_has_anchor_id_like(sql, anchor):
            return _remove_distinct(sql)
        else:
            return _add_distinct(sql)
    else:
        return _add_distinct(sql)

# === ORDER BY policy ===
def _extract_order_by_clause(sql: str) -> Optional[re.Match]:
    return re.search(r"\border\s+by\s+(.*?)(?=(\blimit\b|;|$))", sql, flags=re.I | re.S)

def _remove_order_by_clause(sql: str) -> str:
    return re.sub(r"\s+order\s+by\s+.*?(?=(\blimit\b|;|$))", "", sql, flags=re.I | re.S)

def _split_csv(expr: str) -> List[str]:
    return [p.strip() for p in expr.split(",") if p.strip()]

def _normalize_expr(expr: str) -> str:
    x = re.sub(r"[`\"\s]+", "", expr)
    return x.lower()

def _colname_only(expr: str) -> str:
    parts = re.split(r"\.", expr)
    return parts[-1] if parts else expr

def _select_list_expressions(sql: str) -> List[str]:
    m = re.search(r"^\s*select\s+(.*?)\s+from\s+", sql, flags=re.I | re.S)
    if not m:
        return []
    return _normalize_cols(m.group(1))

def _order_by_is_compatible_with_select_distinct(sql: str) -> bool:
    m_ob = _extract_order_by_clause(sql)
    if not m_ob:
        return True
    ob_items = _split_csv(m_ob.group(1))
    if not ob_items:
        return True
    sel_exprs_raw = _select_list_expressions(sql)
    sel_norm_full = {_normalize_expr(e) for e in sel_exprs_raw}
    sel_norm_colonly = {_normalize_expr(_colname_only(e)) for e in sel_exprs_raw}
    for item in ob_items:
        if re.fullmatch(r"\d+", item.strip()):
            continue
        item_norm_full = _normalize_expr(item)
        item_norm_col = _normalize_expr(_colname_only(item))
        if (item_norm_full not in sel_norm_full) and (item_norm_col not in sel_norm_colonly):
            return False
    return True

def _fix_distinct_order_by(sql: str) -> str:
    if not _has_distinct(sql):
        return sql
    m_ob = _extract_order_by_clause(sql)
    if not m_ob:
        return sql
    if _order_by_is_compatible_with_select_distinct(sql):
        return sql
    sel_exprs = _select_list_expressions(sql)
    if len(sel_exprs) == 1:
        start, end = m_ob.span()
        return sql[:start] + " ORDER BY 1" + sql[end:]
    else:
        return _remove_order_by_clause(sql)

def _question_requests_ordering(question: str) -> bool:
    q = (question or "").lower()
    keywords = [
        "order by", "sort", "sorted", "ascending", "descending", "asc", "desc",
        "alphabetical", "alphabetically", "latest", "newest", "oldest",
        "highest", "lowest", "top", "bottom",
        "urut", "urutkan", "diurutkan", "menaik", "menurun",
        "terbesar", "terkecil", "tertinggi", "terendah", "terbaru", "terlama",
    ]
    return any(k in q for k in keywords)

# ========= Date parsing from question =========
_MONTHS = {
    "january": "01", "february": "02", "march": "03", "april": "04",
    "may": "05", "june": "06", "july": "07", "august": "08",
    "september": "09", "october": "10", "november": "11", "december": "12",
    "jan": "01", "feb": "02", "mar": "03", "apr": "04", "may": "05", "jun": "06",
    "jul": "07", "aug": "08", "sep": "09", "sept": "09", "oct": "10", "nov": "11", "dec": "12",
}
def _pad2(x: str) -> str:
    return x if len(x) == 2 else ("0" + x)[-2:]

def _extract_explicit_dates(question: str) -> Optional[tuple[str, str]]:
    q = (question or "").strip()
    if not q:
        return None
    iso = re.findall(r"\b(\d{4})-(\d{2})-(\d{2})\b", q)
    if len(iso) >= 2:
        d1 = f"{iso[0][0]}-{iso[0][1]}-{iso[0][2]}"
        d2 = f"{iso[1][0]}-{iso[1][1]}-{iso[1][2]}"
        return (d1, d2) if d1 <= d2 else (d2, d1)
    pat1 = re.compile(
        r"\b([A-Za-z]{3,9})\s+(\d{1,2}),\s*(\d{4})\b.*?\b([A-Za-z]{3,9})\s+(\d{1,2}),\s*(\d{4})\b",
        re.I | re.S
    )
    m = pat1.search(q)
    if m:
        m1, d1, y1, m2, d2, y2 = m.groups()
        m1n = _MONTHS.get(m1.lower()); m2n = _MONTHS.get(m2.lower())
        if m1n and m2n:
            s1 = f"{y1}-{m1n}-{_pad2(d1)}"; s2 = f"{y2}-{m2n}-{_pad2(d2)}"
            return (s1, s2) if s1 <= s2 else (s2, s1)
    pat2 = re.compile(
        r"\b(\d{1,2})\s+([A-Za-z]{3,9})\s+(\d{4})\b.*?\b(\d{1,2})\s+([A-Za-z]{3,9})\s+(\d{4})\b",
        re.I | re.S
    )
    m = pat2.search(q)
    if m:
        d1, m1, y1, d2, m2, y2 = m.groups()
        m1n = _MONTHS.get(m1.lower()); m2n = _MONTHS.get(m2.lower())
        if m1n and m2n:
            s1 = f"{y1}-{m1n}-{_pad2(d1)}"; s2 = f"{y2}-{m2n}-{_pad2(d2)}"
            return (s1, s2) if s1 <= s2 else (s2, s1)
    return None

def _rewrite_any_month_between_to_exact(sql: str, start_ymd: str, end_ymd: str) -> str:
    def wrap_date(lhs: str) -> str:
        lhs = lhs.strip()
        return lhs if lhs.upper().startswith("DATE(") else f"DATE({lhs})"
    sql2 = re.sub(
        r"DATE_FORMAT\(\s*(?P<expr>[^)]+?)\s*,\s*'%Y-%m'\s*\)\s+BETWEEN\s+'(?P<m1>\d{4}-\d{2})'\s+AND\s+'(?P<m2>\d{4}-\d{2})'",
        lambda m: f"{wrap_date(m.group('expr'))} BETWEEN '{start_ymd}' AND '{end_ymd}'",
        sql, flags=re.I
    )
    sql2 = re.sub(
        r"(?P<lhs>(?:DATE\s*\(\s*[^)]+\s*\))|(?:[A-Za-z_][\w]*\s*\.\s*[A-Za-z_][\w]*))\s+BETWEEN\s+'(?P<m1>\d{4}-\d{2})'\s+AND\s+'(?P<m2>\d{4}-\d{2})'",
        lambda m: f"{wrap_date(m.group('lhs'))} BETWEEN '{start_ymd}' AND '{end_ymd}'",
        sql2, flags=re.I
    )
    return sql2

def _has_any_date_predicate(sql: str) -> bool:
    m = re.search(r"\bwhere\b(.*?)(?=\border\b|\bgroup\b|\bhaving\b|\blimit\b|;|$)", sql, flags=re.I|re.S)
    if not m:
        return False
    body = m.group(1)
    if re.search(r"\bbetween\b\s*'?\d{4}-\d{2}-\d{2}'?\s*\band\b\s*'?\d{4}-\d{2}-\d{2}'?", body, flags=re.I):
        return True
    if re.search(r"(=|>=|<=|>|<)\s*'?\d{4}-\d{2}-\d{2}'?", body):
        return True
    return False

# ===== Dynamic date column from context (cards) =====
def _extract_date_columns_from_context(context: str, table_short: str) -> List[str]:
    dateish: List[str] = []
    # parse our cards: lines like "TABLE: X", "COLUMN: Y", "TYPE: Z", "DESC: ...", "FQN: ..."
    blocks = [b.strip() for b in context.split("\n\n") if "TABLE:" in b and "COLUMN:" in b]
    for b in blocks:
        t = ""
        c = ""
        dtype = ""
        desc = ""
        for ln in b.splitlines():
            if ln.startswith("TABLE:"):
                t = ln.split(":",1)[1].strip()
            elif ln.startswith("COLUMN:"):
                c = ln.split(":",1)[1].strip()
            elif ln.startswith("TYPE:"):
                dtype = ln.split(":",1)[1].strip().upper()
            elif ln.startswith("DESC:"):
                desc = ln.split(":",1)[1].strip().lower()
        if t.lower() == table_short.lower() and c:
            is_date_type = any(k in dtype for k in ["DATE", "DATETIME", "TIMESTAMP"])
            is_date_desc = "date" in desc
            if is_date_type or is_date_desc:
                dateish.append(c)

    prefer_order = ["orderdate", "requireddate", "shippeddate", "paymentdate",
                    "createdat", "updatedat", "created_on", "updated_on", "created", "updated", "date"]
    def _score(name: str) -> tuple:
        n = name.lower()
        try:
            idx = prefer_order.index(n)
        except ValueError:
            idx = 999
        return (idx, 0 if "date" in n else 1, len(n))

    dateish = list(dict.fromkeys(dateish))
    dateish.sort(key=_score)
    return dateish

def _guess_anchor_date_col_dynamic(sql: str, anchor: str, context: str) -> Optional[str]:
    amap = _build_alias_table_map(sql)
    table_short = amap.get(anchor) or anchor
    cols = _extract_date_columns_from_context(context, table_short)
    return cols[0] if cols else None

def _inject_date_between(sql: str, anchor: str, date_col: str, start_ymd: str, end_ymd: str) -> str:
    lhs = f"DATE({anchor}.{date_col})"
    if re.search(r"\bwhere\b", sql, flags=re.I):
        return re.sub(r";\s*$", "", sql) + f" AND {lhs} BETWEEN '{start_ymd}' AND '{end_ymd}';"
    else:
        return re.sub(r";\s*$", "", sql) + f" WHERE {lhs} BETWEEN '{start_ymd}' AND '{end_ymd}';"

def _remove_meaningless_where(sql: str) -> str:
    m = re.search(r"\bwhere\b(.*?)(?=\border\b|\bgroup\b|\bhaving\b|\blimit\b|;|$)", sql, flags=re.I | re.S)
    if not m:
        return sql
    body = (m.group(1) or "").strip()
    if not body:
        start, end = m.span()
        return sql[:start] + sql[end:]
    if re.search(r"(=|>=|<=|>|<|\bbetween\b|in\s*\(|like\b|is\s+null\b|is\s+not\s+null\b)", body, flags=re.I):
        return sql
    start, end = m.span()
    return sql[:start] + sql[end:]

# === Postprocess orchestration ===
def _collapse_spaces_outside_quotes(s: str) -> str:
    parts = re.split(r"('(?:''|[^'])*')", s)
    for i in range(len(parts)):
        if i % 2 == 0:
            parts[i] = re.sub(r"[ \t]{2,}", " ", parts[i])
    return "".join(parts)

def _postprocess_sql(sql: str, question: str, context: str = "") -> str:
    if not sql:
        return sql
    s = sql.strip()

    def _norm_kw(s_: str) -> str:
        rules = [
            (r"\bselect(?=\S)",          "SELECT "),
            (r"\bfrom(?=\S)",            "FROM "),
            (r"\bwhere(?=\S)",           "WHERE "),
            (r"\bgroup\s*by(?=\S)",      "GROUP BY "),
            (r"\border\s*by(?=\S)",      "ORDER BY "),
            (r"\bhaving(?=\S)",          "HAVING "),
            (r"\blimit(?=\S)",           "LIMIT "),
            (r"\binner\s*join(?=\S)",    "INNER JOIN "),
            (r"\bleft\s*join(?=\S)",     "LEFT JOIN "),
            (r"\bright\s*join(?=\S)",    "RIGHT JOIN "),
            (r"\bfull\s*join(?=\S)",     "FULL JOIN "),
            (r"\bjoin(?=\S)",            "JOIN "),
            (r"\bon(?=\S)",              "ON "),
        ]
        out = s_
        for pat, rep in rules:
            out = re.sub(pat, rep, out, flags=re.I)
        out = re.sub(r"[ \t]+", " ", out)
        return out

    s = _norm_kw(s)
    if not s.endswith(";"):
        s += ";"

    exact = _extract_explicit_dates(question)
    if exact:
        s = _rewrite_any_month_between_to_exact(s, exact[0], exact[1])

    s = _fix_bad_backticks(s)
    s = _pushdown_filters_through_equi_join(s)
    s = _remove_redundant_joins(s)
    s = _prefer_distinct_over_group_by(s)
    s = _drop_unnecessary_distinct(s)
    s = _strip_where_non_anchor_refs(s)

    s = _strip_group_by_having_non_anchor(s)
    s = _strip_lonely_having(s)
    s = _trim_dangling_closing_paren(s)

    s = _fix_missing_space_before_clauses(s)

    anchor = _find_anchor_qualifier(s)
    select_quals = _select_qualifiers(s)
    if anchor and select_quals:
        non_anchor_quals = select_quals - {anchor}
        has_filtering = _has_filters(s)
        if not non_anchor_quals and not has_filtering and _has_join(s):
            s = _strip_all_joins_unconditionally(s)

    s = _ensure_reasonable_distinct(s, question)

    ob_match = _extract_order_by_clause(s)
    if ob_match:
        if not _question_requests_ordering(question):
            s = _remove_order_by_clause(s)
        else:
            s = _fix_distinct_order_by(s)

    s = _maybe_add_limit(question, s)

    s = _fix_missing_space_before_clauses(s)
    s = _collapse_spaces_outside_quotes(s)

    if exact and not _has_any_date_predicate(s):
        anchor2 = _find_anchor_qualifier(s)
        if anchor2:
            dyn_col = _guess_anchor_date_col_dynamic(s, anchor2, context)
            if dyn_col:
                s = _inject_date_between(s, anchor2, dyn_col, exact[0], exact[1])

    s = _remove_meaningless_where(s)

    if not re.search(r"\bselect\b", s, flags=re.I):
        s = _norm_kw(sql.strip())
        if not s.endswith(";"):
            s += ";"
        s = _fix_missing_space_before_clauses(s)
        s = _collapse_spaces_outside_quotes(s)

    return s

# ---------- Whitelist parsing + validation ----------
_ALNUM = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")

def _extract_internal_hint(ctx: str) -> Dict[str, Dict[str, str]]:
    m = re.search(r"### _INTERNAL_HINT\s+({.*})\s*$", ctx, flags=re.S)
    if not m:
        return {}
    try:
        obj = json.loads(m.group(1))
        return obj.get("alias_map") or {}
    except Exception:
        return {}

def _parse_allowed_identifiers(ctx: str) -> Dict[str, set]:
    tables: Dict[str, set] = {}
    m = re.search(r"### ALLOWED IDENTIFIERS(.*?)(?:\n### |\Z)", ctx, flags=re.S)
    if not m:
        return tables
    block = m.group(1)
    for line in block.splitlines():
        line = line.strip()
        if not line.startswith("- "):
            continue
        parts = line[2:].split(":", 1)
        if len(parts) != 2:
            continue
        table = parts[0].strip().strip("`")
        cols = [c.strip().strip("`") for c in parts[1].split(",") if c.strip()]
        if table:
            tables.setdefault(table, set()).update(cols)
    return tables

def _identify_used_identifiers(sql: str) -> Dict[Optional[str], set]:
    qualified = re.findall(r"`?([A-Za-z_][A-Za-z0-9_]*)`?\s*\.\s*`?([A-Za-z_][A-Za-z0-9_]*)`?", sql)
    out: Dict[Optional[str], set] = {}
    for t, c in qualified:
        out.setdefault(t, set()).add(c)
    keywords = {"select","from","where","group","by","order","limit","join","left","right","inner","on","and","or","not","as","distinct","count","sum","avg","min","max","having","asc","desc","like","in","between","is","null","case","when","then","else","end"}
    for tok in re.findall(r"\b([A-Za-z_][A-Za-z0-9_]*)\b", sql):
        if tok.lower() in keywords:
            continue
        out.setdefault(None, set()).add(tok)
    return out

def _validate_identifiers(sql: str, ctx: str) -> List[str]:
    allowed = _parse_allowed_identifiers(ctx)
    used = _identify_used_identifiers(sql)
    errs: List[str] = []
    for tbl, cols in used.items():
        if tbl is None:
            continue
        if tbl not in allowed:
            errs.append(f"table `{tbl}` not in ALLOWED")
            continue
        bad = [c for c in cols if c not in allowed[tbl]]
        if bad:
            errs.append(f"{tbl} -> invalid columns: {', '.join(bad)}")
    return errs

def _repair_with_aliases(sql: str, ctx: str) -> str:
    alias_map_per_table = _extract_internal_hint(ctx)
    if not alias_map_per_table:
        return sql
    global_map: Dict[str, Optional[str]] = {}
    for _t, amap in alias_map_per_table.items():
        for al, canon in amap.items():
            al = str(al).lower()
            if al not in global_map:
                global_map[al] = canon
            elif global_map[al] != canon:
                global_map[al] = None  # ambiguous
    keywords = {"select","from","where","group","by","order","limit","join","left","right","inner","on","and","or","not","as","distinct","count","sum","avg","min","max","having","asc","desc","like","in","between","is","null","case","when","then","else","end"}
    pattern = re.compile(r"\b([A-Za-z_][A-Za-z0-9_]*)\b")
    def smart_repl(m):
        w = m.group(0)
        lw = w.lower()
        if lw in keywords:
            return w
        canon = global_map.get(lw)
        return canon if canon and _ALNUM.match(canon) else w
    return pattern.sub(smart_repl, sql)

# ---------- Prompt ----------
def _make_generation_prompt(ctx: str, question: str) -> str:
    return f"""
You are a senior SQL assistant for MySQL 8.0.

Return a runnable query using ONLY tables/columns listed in the ALLOWED IDENTIFIERS section of Context.
If a needed column name appears as an alias in Context, use its canonical column name.

########################
# STRICT OUTPUT FORMAT #
########################
{{ "sql": "<final MySQL query>", "notes": "<brief rationale; for EACH JOIN, name the exact selected/grouped/filtered column it contributes>" }}
# No extra keys. No Markdown. End SQL with a semicolon.
# If the final SQL has NO JOINs, set "notes" exactly to: "No joins used."

##################
# SQL CONSTRAINT #
##################
- MySQL 8.0; backticks only when needed; ANSI joins.

########################################
# ANCHOR TABLE & MINIMAL-JOIN POLICY   #
########################################
- Pick as ANCHOR the table containing the most required columns.
- If the ANCHOR already has a needed field by name/description, USE IT. Do NOT JOIN another table for a duplicate.
- Only JOIN if you SELECT, GROUP BY, or WHERE on a column that is NOT in the ANCHOR.
- If an equi-join ties ANCHOR.col = DIM.col and WHERE only constrains DIM.col with a literal, PUSH the filter to ANCHOR.col and REMOVE the JOIN unless DIM also contributes a selected/grouped column.
- After drafting, AUDIT each JOIN: if you cannot state which output/filter column comes from that table, REMOVE it.

########################################
# DISTINCT & GROUP BY
########################################
- Avoid DISTINCT unless: (a) explicitly asked, or (b) a JOIN creates duplicates in output columns.
- If dedup without aggregation is intended, prefer SELECT DISTINCT over GROUP BY; otherwise avoid both.

########################################
# DATE BUCKETING (MySQL-safe)
########################################
- DATE(dt) AS day; DATE_FORMAT(dt,'%Y-%m') AS month; CONCAT(YEAR(dt),'-Q',QUARTER(dt)) AS quarter; YEAR(dt) AS year.
- Always GROUP BY any selected bucket/dimension.

#################################
# FILTERS & TYPES
#################################
- Do not invent date ranges. Cast textual dates with STR_TO_DATE if needed.
- Only add WHERE conditions clearly implied by the Question.

################
# CONTEXT
################
{ctx}

Question:
{question}
""".strip()

# ---------- Notes sanitizer ----------
def _sanitize_notes(sql: str, notes: str) -> str:
    if not notes:
        return notes
    if not _has_join(sql or ""):
        return "No joins used."
    return notes.strip()

# ---------- Generation (with validation & regeneration) ----------
def generate_node(state: RAGState) -> RAGState:
    llm = ChatOllama(
        model=CHAT_MODEL,
        base_url=OLLAMA_BASE_URL,
        temperature=0.0,
        seed=42
    )
    ctx = state.get("context", "")
    question = state.get("question", "")
    prompt = _make_generation_prompt(ctx, question)
    resp = llm.invoke(prompt)
    content = getattr(resp, "content", None) or str(resp)

    answer_out = content
    draft_sql = ""
    final_sql = ""
    try:
        obj = json.loads(content)
        draft_sql = (obj.get("sql") or "").strip()
        notes = (obj.get("notes") or "").strip()

        # DEBUG #1
        if draft_sql:
            _dbg_print_two("Original RAG and generation SQL CODE :", draft_sql)

        # Post-process + alias repair + post-process again
        final_sql = _postprocess_sql(draft_sql, question, ctx) if draft_sql else draft_sql
        if final_sql:
            final_sql = _repair_with_aliases(final_sql, ctx)
            final_sql = _postprocess_sql(final_sql, question, ctx)

        # Validate identifiers against whitelist
        violations = _validate_identifiers(final_sql or "", ctx)
        if violations:
            force_msg = (
                "The previous SQL used identifiers not present in ALLOWED IDENTIFIERS:\n- "
                + "\n- ".join(violations)
                + "\nRewrite the SQL using ONLY identifiers listed in ALLOWED IDENTIFIERS. "
                  "If a synonym appears, choose the canonical column."
            )
            strict_prompt = _make_generation_prompt(ctx + "\n\n" + force_msg, question)
            resp2 = llm.invoke(strict_prompt)
            content2 = getattr(resp2, "content", None) or str(resp2)
            try:
                obj2 = json.loads(content2)
                draft2 = (obj2.get("sql") or "").strip()
                notes2 = (obj2.get("notes") or "").strip()
            except Exception:
                draft2, notes2 = content2, ""
            sql2 = _postprocess_sql(draft2 or "", question, ctx) if draft2 else ""
            if sql2:
                sql2 = _repair_with_aliases(sql2, ctx)
                sql2 = _postprocess_sql(sql2, question, ctx)
            v1 = len(violations)
            v2 = len(_validate_identifiers(sql2 or "", ctx))
            if v2 <= v1 and sql2:
                final_sql, notes = sql2, (notes2 or notes)

        # DEBUG #2
        if final_sql:
            _dbg_print_two("After sanitzie SQL CODE:", final_sql)

        # Minimalist enforcement for simple questions (keep from your original)
        question_lower = question.lower()
        simple_keywords = ["show", "list", "get", "what are", "display", "give me"]
        complex_keywords = ["who ordered", "with orders", "placed", "purchased", "bought", "having"]
        is_simple_query = any(k in question_lower for k in simple_keywords)
        is_complex_query = any(k in question_lower for k in complex_keywords)
        if is_simple_query and not is_complex_query and " join " in (" " + (final_sql or "").lower() + " "):
            final_sql = _strip_all_joins_unconditionally(final_sql)

        # Notes sanitizer aligned with final SQL
        notes = _sanitize_notes(final_sql or "", notes or "")

        if final_sql:
            answer_out = final_sql + ("\n\n-- notes: " + notes if notes else "")
        else:
            answer_out = (notes or "").strip() or content
    except Exception:
        pass

    state["answer"] = answer_out
    state["draft_sql"] = draft_sql
    state["final_sql"] = final_sql
    return state

def build_app():
    g = StateGraph(RAGState)
    g.add_node("ingest", ingest_node)
    g.add_node("retrieve", retrieve_node)
    g.add_node("generate", generate_node)
    g.set_entry_point("ingest")
    g.add_edge("ingest", "retrieve")
    g.add_edge("retrieve", "generate")
    g.add_edge("generate", END)
    return g.compile()

# =========================
# === MCP UTIL ============
# =========================
async def _mcp_call(client: Client, tool: str, payload: dict | None = None, timeout: int = MCP_TIMEOUT):
    coro = client.call_tool(tool, payload or {})
    return await asyncio.wait_for(coro, timeout=timeout)

def _extract_payload(result) -> Any:
    if hasattr(result, 'data') and result.data is not None:
        return result.data
    if hasattr(result, "structuredContent") and result.structuredContent is not None:
        return result.structuredContent
    if hasattr(result, "content") and result.content:
        return result.content[0].text if isinstance(result.content, list) else str(result.content)
    return result

def _ensure_json(d: Any) -> Optional[dict]:
    if isinstance(d, dict):
        return d
    if isinstance(d, str):
        try:
            return json.loads(d)
        except Exception:
            return None
    return None

def display_tables_list(data: dict):
    tables = data.get("tables", [])
    total = data.get("total_tables", len(tables))
    if not tables:
        print("No tables found.")
        return
    print("\nAvailable tables:")
    for i, t in enumerate(tables, start=1):
        print(f"  {i:>2}. {t}")
    print(f"\nTotal: {total} table(s)")

def display_table_from_json(data):
    preview_rows = data.get("preview_rows", [])
    preview_cols = data.get("preview_columns", [])
    total_rows = data.get("total_rows", len(preview_rows))
    total_cols = data.get("total_columns", len(preview_cols))
    if preview_rows and preview_cols:
        df = pd.DataFrame(preview_rows, columns=preview_cols)
        df.index = df.index + 1
        if total_rows > len(preview_rows):
            df.loc["..."] = ["..."] * len(df.columns)
        if total_cols > len(df.columns):
            df["..."] = "..."
        print("\nQuery Result (preview):")
        print(tabulate(df, headers="keys", tablefmt="pretty"))
        print(f"\nShowing preview: {len(preview_rows)} rows × {len(preview_cols)} columns "
              f"(full result: {total_rows} rows × {total_cols} columns)")
        dbg = data.get("_debug_info", {}) or {}
        if dbg:
            print(f"Debug: Query returned {dbg.get('query_returned_rows', '?')} rows, "
                  f"True total: {dbg.get('true_total_rows', '?')} rows")
    else:
        print("Query executed but returned no rows.")

def _save_full_to_csv(columns: list, rows: list, suggested_name: str | None, output_dir: str) -> str:
    os.makedirs(output_dir, exist_ok=True)
    df = pd.DataFrame(rows, columns=columns)
    stamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    base = suggested_name or "export"
    fname = f"export_{base}_{stamp}.csv"
    full_path = os.path.join(output_dir, fname)
    df.to_csv(full_path, index=False)
    return os.path.abspath(full_path)

def _jupyter_files_url(abs_path: str) -> Optional[str]:
    try:
        root = os.getcwd()
        rel = os.path.relpath(abs_path, root)
        if rel.startswith(".."):
            return None
        return "/files/" + rel.replace(os.sep, "/")
    except Exception:
        return None

# ======== CSV-ONLY EXPORT HELPER ========
async def _maybe_offer_export(mcp: Client, payload: Optional[dict], answer_sql: str):
    if not isinstance(payload, dict) or not payload.get("success"):
        return
    yn = input("\nExport FULL result to CSV? [y/N]: ").strip().lower()
    if yn not in ("y", "yes"):
        return
    try:
        dbg = payload.get("_debug_info", {}) or {}
        base_query = dbg.get("base_query") or answer_sql.split("\n\n-- notes:")[0].strip().rstrip(";")
        full = await _mcp_call(mcp, "query_db_full", {"sql": base_query})
        full_payload = _ensure_json(_extract_payload(full))
        if isinstance(full_payload, dict) and full_payload.get("success"):
            cols = full_payload.get("columns", [])
            rows = full_payload.get("rows", [])
            os.makedirs(EXPORT_DIR, exist_ok=True)
            m = re.search(r'from\s+("?([A-Za-z0-9_]+)"?\.)?"?([A-Za-z0-9_]+)"?', base_query, flags=re.I)
            suggested = m.group(3) if m else "export"
            outpath = _save_full_to_csv(cols, rows, suggested, EXPORT_DIR)
            print(f"Saved CSV: {outpath}")
            rel = _jupyter_files_url(outpath)
            if rel:
                url = f"{JUPYTER_BASE_URL}{rel}" if JUPYTER_BASE_URL else rel
                print(f"Download via Jupyter: {url}")
            if full_payload.get("truncated"):
                print("Export was truncated to server cap.")
        else:
            print("Failed to export full result.")
    except Exception as e:
        print(f"Export error: {e}")

# =========================
# === REPL RUNNER =========
# =========================
async def repl(paths: List[str]):
    app = build_app()
    show_graph_mermaid_only(app)

    transport = StreamableHttpTransport(MCP_BASE)
    async with Client(transport=transport) as mcp:
        try:
            tools = await mcp.list_tools()
            names = [getattr(t, "name", None) for t in tools]
            print("\nMCP tools:", ", ".join([n for n in names if n]) or "(none)")
        except Exception as e:
            print(f"Could not list MCP tools: {e}")

        first_q = input("Enter a test question (or press Enter to skip): ").strip()
        if not first_q:
            first_q = "Which columns describe customer identity?"

        init: RAGState = {
            "jsonl_paths": paths,
            "question": first_q,
            "added": 0,
            "updated": 0,
            "skipped": 0,
            "context": "",
            "answer": "",
            "draft_sql": "",
            "final_sql": "",
        }

        out = app.invoke(init)

        # RAG ingest report
        vs = build_vector_store()
        total = vs._collection.count()
        print("\n=== RAG Ingest Report ===")
        print(f"Added (first run): {out.get('added', 0)}")
        print(f"Updated (upserted): {out.get('updated', 0)}")
        print(f"Skipped (unchanged): {out.get('skipped', 0)}")
        print(f"Total docs now in collection: {total}")
        print(f"Chroma path: {Path(CHROMA_DIR).resolve()} | Collection: {COLLECTION_NAME}")

        print("\n--- Generated Answer (RAG) ---\n")
        generated = out.get("answer", "(no answer)")
        print(generated)

        async def exec_via_mcp(sql_block: str, final_sql: Optional[str] = None):
            sql_only = sql_block.split("\n\n-- notes:")[0].strip()
            sql_clean = (final_sql or sql_only).rstrip(";").strip()

            try:
                lt = await _mcp_call(mcp, "list_tables", {})
                lt_payload = _ensure_json(_extract_payload(lt)) or {}
                if lt_payload.get("success"):
                    display_tables_list(lt_payload)
            except Exception as e:
                print(f"list_tables failed (skip): {e}")

            print(f"\nSQL (RAG -> MCP):\n{sql_clean};")

            try:
                res = await _mcp_call(mcp, "query_db", {"sql": sql_clean})
                payload = _ensure_json(_extract_payload(res))
                if isinstance(payload, dict) and payload.get("success"):
                    display_table_from_json(payload)
                    return payload
                else:
                    if isinstance(payload, dict) and payload.get("suggestions"):
                        print("\nTable not found. Suggestions:")
                        for s in payload["suggestions"]:
                            print(" -", s)
                    if isinstance(payload, dict) and payload.get("column_suggestions"):
                        print("\nColumn suggestions:")
                        for s in payload["column_suggestions"]:
                            print(" -", s)
                    err = payload.get("error") if isinstance(payload, dict) else str(payload)
                    print(f"query_db failed: {err}")
                    return None
            except asyncio.TimeoutError:
                print("MCP query_db timeout")
                return None
            except Exception as e:
                print(f"MCP query_db error: {e}")
                return None

        payload = await exec_via_mcp(generated, out.get("final_sql"))
        await _maybe_offer_export(mcp, payload, generated)

        print("\nAsk me more questions! (type 'exit' to quit)")
        while True:
            q = input("\nQuestion> ").strip()
            if not q or q.lower() in {"exit", "quit", "q"}:
                print("Bye.")
                break

            state: RAGState = {
                "jsonl_paths": paths,
                "question": q,
                "added": 0,
                "updated": 0,
                "skipped": 0,
                "context": "",
                "answer": "",
                "draft_sql": "",
                "final_sql": "",
            }
            state = retrieve_node(state)
            state = generate_node(state)

            print("\n--- Generated Answer (RAG) ---\n")
            print(state.get("answer", "(no answer)"))

            payload = await exec_via_mcp(state.get("answer", ""), state.get("final_sql"))
            await _maybe_offer_export(mcp, payload, state.get("answer", ""))

def main():
    paths = resolve_jsonl_paths(JSON_INPUT)
    if not paths:
        raise RuntimeError(
            "No JSONL inputs found.\n"
            "Set EXPORT_JSONL_OUTPUT to one of:\n"
            "  • a single file: ./exports/classicmodels.jsonl\n"
            "  • a directory:   ./datahub_metadata\n"
            "  • a glob:        ./datahub_metadata/*.jsonl\n"
        )
    print("JSONL inputs:")
    for p in paths:
        print(" -", p)

    try:
        asyncio.run(repl(paths))
    except RuntimeError as e:
        if "asyncio.run() cannot be called" in str(e):
            import nest_asyncio
            nest_asyncio.apply()
            loop = asyncio.get_event_loop()
            loop.run_until_complete(repl(paths))
        else:
            raise

if __name__ == "__main__":
    main()

JSONL inputs:
 - metadata_jsonl/classicmodels.jsonl



MCP tools: list_tables, query_db, query_db_full


Enter a test question (or press Enter to skip):  List all customer names (alphabetical).



=== RAG Ingest Report ===
Added (first run): 0
Updated (upserted): 0
Skipped (unchanged): 59
Total docs now in collection: 59
Chroma path: /app/chroma_db | Collection: classicmodels

--- Generated Answer (RAG) ---

 SELECT c.customerName FROM customers c ORDER BY c.customerName;

Available tables:
   1. classicmodels.customers
   2. classicmodels.employees
   3. classicmodels.offices
   4. classicmodels.orderdetails
   5. classicmodels.orders
   6. classicmodels.payments
   7. classicmodels.productlines
   8. classicmodels.products
   9. datahub.metadata_aspect_v2

Total: 9 table(s)

SQL (RAG -> MCP):
SELECT c.customerName FROM customers c ORDER BY c.customerName;

Query Result (preview):
+-----+-------------------------+
|     |      customerName       |
+-----+-------------------------+
|  1  |      Alpha Cognac       |
|  2  | American Souvenirs Inc  |
|  3  |   Amica Models & Co.    |
|  4  |      ANG Resellers      |
|  5  | Anna's Decorations, Ltd |
| ... |           ...        


Export FULL result to CSV? [y/N]:  n



Ask me more questions! (type 'exit' to quit)



Question>  exit


Bye.
